
# Лабораторная 3, вариант 2: Трансформер RU → EN (версия для Colab Pro)

**Эта версия отличается от локального `Lab3.ipynb`:**
- Больше данных (`N_TRAIN = 400_000` против 200K — почти полный OPUS-100 ru-en после фильтров).
- Больший словарь (`VOCAB_SIZE = 32_000`).
- Конфиг Transformer-**base**: `d_model = 512`, `dim_ff = 2048`, ≈63 M параметров (против 19 M локально).
- Авто-детект GPU (L4 24 GB / A100 40 GB) с подгонкой `BATCH`.
- Опциональный mount Google Drive для сохранения чекпоинтов между сессиями.

Цель — поднять BLEU с ~22 (локально) до 25–28 за ~1 час на L4.

**Перед запуском:** *Runtime → Change runtime type → GPU → L4 (или A100)*.



## Установка зависимостей

`torch`, `datasets`, `tokenizers`, `matplotlib`, `tqdm` уже стоят в Colab. Доустанавливаем только `sacrebleu`.


In [ ]:

!pip install -q -U sacrebleu datasets tokenizers



## Проверка GPU и автоподбор `BATCH`

Colab Pro выдаёт разные GPU (T4 16 GB / L4 24 GB / A100 40 GB) в зависимости от загрузки. Подбираем `BATCH` под доступную память.


In [ ]:

import torch

assert torch.cuda.is_available(), 'Включите GPU: Runtime → Change runtime type → GPU'

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_ok = torch.cuda.is_bf16_supported()

print(f'GPU: {gpu_name}')
print(f'память: {gpu_mem_gb:.1f} GB')
print(f'bf16: {bf16_ok}')

if gpu_mem_gb >= 35:        # A100 40 GB
    AUTO_BATCH = 128
elif gpu_mem_gb >= 22:      # L4 / RTX A5000 24 GB
    AUTO_BATCH = 64
elif gpu_mem_gb >= 14:      # T4 16 GB
    AUTO_BATCH = 32
else:
    AUTO_BATCH = 16

print(f'AUTO_BATCH = {AUTO_BATCH}')



## Google Drive (опционально)

Если хотите сохранять чекпоинты между сессиями Colab — поставьте `USE_DRIVE = True` ниже и подтвердите доступ. Иначе всё пишется в `/content/Lab3/` и пропадёт после отключения runtime'а.


In [ ]:

USE_DRIVE = False    # True — монтируем Drive; иначе работаем в эфемерном /content/Lab3

from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Lab3')
else:
    ROOT = Path('/content/Lab3')

ROOT.mkdir(parents=True, exist_ok=True)
print('рабочая директория:', ROOT)



## Импорты и инициализация


In [ ]:

import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import json
import math
import random
import re

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda')



## Гиперпараметры

Конфиг Transformer-**base** из оригинальной статьи — `d_model=512`, `ff=2048`, 6/6 слоёв, 8 голов. Около 63 M параметров.


In [ ]:

SRC_LANG, TGT_LANG = 'ru', 'en'

# Данные
MAX_WS_LEN  = 64
MAX_BPE_LEN = 96
N_TRAIN  = 400_000      # vs 200K локально — почти весь полезный OPUS-100 после фильтров
N_VAL    = 2_000
N_TEST   = 2_000

# Токенизатор
VOCAB_SIZE = 32_000     # увеличен под больший корпус и больший d_model

# Модель — Transformer-base
D_MODEL    = 512
NHEAD      = 8
N_ENC      = 6
N_DEC      = 6
DIM_FF     = 2048
DROPOUT    = 0.1
NORM_FIRST = True

# Обучение
BATCH        = AUTO_BATCH
EPOCHS       = 5
WARMUP       = 6_000
LABEL_SMOOTH = 0.1
CLIP         = 0.5
ADAM_EPS     = 1e-8
USE_AMP      = True
AMP_DTYPE    = torch.bfloat16 if bf16_ok else torch.float16

# Декодирование
BEAM = 5

# Пути
TOKENIZER_FP  = ROOT / 'tokenizer.json'
BEST_CKPT     = ROOT / 'best.pth'
LAST_CKPT     = ROOT / 'last.pth'
HISTORY_JSON  = ROOT / 'history.json'

print(f'BATCH={BATCH}  D_MODEL={D_MODEL}  N_TRAIN={N_TRAIN}  VOCAB={VOCAB_SIZE}  EPOCHS={EPOCHS}')



## Данные: OPUS-100 (ru-en)

Берём полный `train` (1 M пар), фильтруем:

1. пустые пары;
2. **мусор** — строки с шаблоном `<<<…>>>` и пары, где меньше 50 % символов — буквы (в OPUS-100 много шума);
3. слишком длинные пары (> `MAX_WS_LEN` whitespace-токенов).

Затем перемешиваем и режем по `N_TRAIN` / `N_VAL` / `N_TEST`.


In [ ]:

from datasets import load_dataset

raw = load_dataset('Helsinki-NLP/opus-100', 'en-ru')
print({split: len(raw[split]) for split in raw})

GARBAGE_PAT = re.compile(r'<<<|>>>|^[\W\d_]+$|^\s*$')

def is_garbage(text):
    t = text.strip()
    if not t:
        return True
    if GARBAGE_PAT.search(t):
        return True
    letters = sum(1 for ch in t if ch.isalpha())
    if letters / max(len(t), 1) < 0.5:
        return True
    return False

def flatten(split):
    pairs = []
    for ex in split:
        en = ex['translation']['en'].strip()
        ru = ex['translation']['ru'].strip()
        if not en or not ru:
            continue
        if is_garbage(en) or is_garbage(ru):
            continue
        if len(en.split()) > MAX_WS_LEN or len(ru.split()) > MAX_WS_LEN:
            continue
        pairs.append((ru, en))
    return pairs

rng = random.Random(SEED)

train_all = flatten(raw['train'])
rng.shuffle(train_all)
train_pairs = train_all[:N_TRAIN]

val_all = flatten(raw['validation'])
rng.shuffle(val_all)
val_pairs = val_all[:N_VAL]

test_all = flatten(raw['test'])
rng.shuffle(test_all)
test_pairs = test_all[:N_TEST]

print(f'train: {len(train_pairs)} | val: {len(val_pairs)} | test: {len(test_pairs)}')
for ru, en in train_pairs[:3]:
    print(f'RU: {ru}')
    print(f'EN: {en}')
    print('-' * 60)



## Токенизатор: общий BPE

32K мерджей над объединённым RU+EN корпусом. Спецтокены: `[PAD] [UNK] [BOS] [EOS]`. Если файл уже сохранён — подгружаем.


In [ ]:

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

SPECIALS = ['[PAD]', '[UNK]', '[BOS]', '[EOS]']

if TOKENIZER_FP.exists():
    tok = Tokenizer.from_file(str(TOKENIZER_FP))
    print(f'Загружен токенизатор из {TOKENIZER_FP}')
else:
    tok = Tokenizer(models.BPE(unk_token='[UNK]'))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()

    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=SPECIALS,
        min_frequency=2,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )

    def iter_train():
        for ru, en in train_pairs:
            yield ru
            yield en

    tok.train_from_iterator(iter_train(), trainer=trainer, length=2 * len(train_pairs))
    tok.save(str(TOKENIZER_FP))
    print(f'Сохранён токенизатор -> {TOKENIZER_FP}')

PAD_ID = tok.token_to_id('[PAD]')
UNK_ID = tok.token_to_id('[UNK]')
BOS_ID = tok.token_to_id('[BOS]')
EOS_ID = tok.token_to_id('[EOS]')
VOCAB  = tok.get_vocab_size()
print(f'vocab={VOCAB} | PAD={PAD_ID} UNK={UNK_ID} BOS={BOS_ID} EOS={EOS_ID}')



## Dataset и DataLoader

Финальный фильтр по BPE-длине, затем стандартные `Dataset` + `collate_fn` с pad-масками.


In [ ]:

def encode_src(text):
    return tok.encode(text).ids + [EOS_ID]

def encode_tgt(text):
    return [BOS_ID] + tok.encode(text).ids + [EOS_ID]


def filter_by_bpe(pairs, max_len=MAX_BPE_LEN):
    kept = []
    for ru, en in tqdm(pairs, desc='filter bpe'):
        if len(encode_src(ru)) > max_len:
            continue
        if len(encode_tgt(en)) > max_len:
            continue
        kept.append((ru, en))
    return kept

train_pairs = filter_by_bpe(train_pairs)
val_pairs   = filter_by_bpe(val_pairs)
test_pairs  = filter_by_bpe(test_pairs)
print(f'после BPE-фильтра: train={len(train_pairs)} | val={len(val_pairs)} | test={len(test_pairs)}')


class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        ru, en = self.pairs[idx]
        return (torch.tensor(encode_src(ru), dtype=torch.long),
                torch.tensor(encode_tgt(en), dtype=torch.long))


def collate(batch):
    srcs, tgts = zip(*batch)
    src = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD_ID)
    tgt = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD_ID)
    tgt_in  = tgt[:, :-1]
    tgt_out = tgt[:, 1:]
    src_pad_mask = (src == PAD_ID)
    tgt_pad_mask = (tgt_in == PAD_ID)
    return src, tgt_in, tgt_out, src_pad_mask, tgt_pad_mask


train_ds = TranslationDataset(train_pairs)
val_ds   = TranslationDataset(val_pairs)
test_ds  = TranslationDataset(test_pairs)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          collate_fn=collate, num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          collate_fn=collate, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,
                          collate_fn=collate, num_workers=2, pin_memory=True)

sample = next(iter(train_loader))
print('shapes:', [tuple(t.shape) for t in sample[:3]])



## Модель: `torch.nn.Transformer` (pre-norm, base)

Всё как в локальной версии, только `d_model=512`, `dim_ff=2048` и `norm_first=True` (pre-norm для устойчивости).


In [ ]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


def generate_square_subsequent_mask(sz, device):
    return torch.triu(torch.ones(sz, sz, dtype=torch.bool, device=device), diagonal=1)


class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab, d_model, nhead, n_enc, n_dec, dim_ff, dropout, pad_id, norm_first=True):
        super().__init__()
        self.d_model = d_model
        self.pad_id  = pad_id
        self.tok_emb = nn.Embedding(vocab, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=n_enc, num_decoder_layers=n_dec,
            dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True, norm_first=norm_first,
        )
        self.proj = nn.Linear(d_model, vocab)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def embed(self, x):
        return self.pos_enc(self.tok_emb(x) * math.sqrt(self.d_model))

    def encode(self, src, src_pad_mask):
        return self.transformer.encoder(self.embed(src), src_key_padding_mask=src_pad_mask)

    def decode(self, tgt, memory, tgt_mask, tgt_pad_mask, src_pad_mask):
        return self.transformer.decoder(
            self.embed(tgt), memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
        )

    def forward(self, src, tgt_in, src_pad_mask, tgt_pad_mask):
        T = tgt_in.size(1)
        tgt_mask = generate_square_subsequent_mask(T, src.device)
        memory = self.encode(src, src_pad_mask)
        dec = self.decode(tgt_in, memory, tgt_mask, tgt_pad_mask, src_pad_mask)
        return self.proj(dec)


model = Seq2SeqTransformer(VOCAB, D_MODEL, NHEAD, N_ENC, N_DEC, DIM_FF, DROPOUT, PAD_ID,
                           norm_first=NORM_FIRST).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'параметров: {n_params/1e6:.2f}M  (norm_first={NORM_FIRST})')

with torch.no_grad():
    _src = torch.randint(4, VOCAB, (2, 10), device=device)
    _tgt = torch.randint(4, VOCAB, (2, 8),  device=device)
    _out = model(_src, _tgt,
                 src_pad_mask=torch.zeros_like(_src, dtype=torch.bool),
                 tgt_pad_mask=torch.zeros_like(_tgt, dtype=torch.bool))
    assert _out.shape == (2, 8, VOCAB), _out.shape
print('forward OK, shape:', tuple(_out.shape))



## Обучение

То же, что и локально: `AdamW` + Noam-расписание + label smoothing + grad clip + AMP (bf16 на L4/A100). `best.pth` обновляется при улучшении val-loss. Если он уже существует — обучение пропускается.


In [ ]:

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=LABEL_SMOOTH)
optimizer = optim.AdamW(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=ADAM_EPS)

def noam_lr(step):
    step = max(step, 1)
    return (D_MODEL ** -0.5) * min(step ** -0.5, step * (WARMUP ** -1.5))

scheduler = LambdaLR(optimizer, lr_lambda=noam_lr)
scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and AMP_DTYPE == torch.float16))


def train_epoch(epoch):
    model.train()
    total, n = 0.0, 0
    pbar = tqdm(train_loader, desc=f'эпоха {epoch} train')
    for src, tgt_in, tgt_out, src_pm, tgt_pm in pbar:
        src, tgt_in, tgt_out = src.to(device, non_blocking=True), tgt_in.to(device, non_blocking=True), tgt_out.to(device, non_blocking=True)
        src_pm, tgt_pm = src_pm.to(device, non_blocking=True), tgt_pm.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(src, tgt_in, src_pm, tgt_pm)
            loss = criterion(logits.reshape(-1, VOCAB), tgt_out.reshape(-1))

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            optimizer.step()
        scheduler.step()

        bs = src.size(0)
        total += loss.item() * bs; n += bs
        pbar.set_postfix(loss=f'{total/n:.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')
    return total / n


@torch.no_grad()
def evaluate(loader, name='val'):
    model.eval()
    total, n = 0.0, 0
    for src, tgt_in, tgt_out, src_pm, tgt_pm in tqdm(loader, desc=name):
        src, tgt_in, tgt_out = src.to(device, non_blocking=True), tgt_in.to(device, non_blocking=True), tgt_out.to(device, non_blocking=True)
        src_pm, tgt_pm = src_pm.to(device, non_blocking=True), tgt_pm.to(device, non_blocking=True)
        with torch.amp.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(src, tgt_in, src_pm, tgt_pm)
            loss = criterion(logits.reshape(-1, VOCAB), tgt_out.reshape(-1))
        bs = src.size(0)
        total += loss.item() * bs; n += bs
    return total / n


if BEST_CKPT.exists() and HISTORY_JSON.exists():
    print(f'Найден чекпоинт {BEST_CKPT} — обучение пропущено.')
    print('Удалите best.pth и history.json для повторного запуска.')
    model.load_state_dict(torch.load(BEST_CKPT, map_location=device, weights_only=True))
else:
    history = {'train_loss': [], 'val_loss': []}
    best_val = float('inf')
    for epoch in range(1, EPOCHS + 1):
        tr = train_epoch(epoch)
        torch.cuda.empty_cache()
        vl = evaluate(val_loader, name=f'эпоха {epoch} val')
        torch.cuda.empty_cache()
        history['train_loss'].append(tr)
        history['val_loss'].append(vl)
        print(f'эпоха {epoch}: train={tr:.4f} | val={vl:.4f}')
        torch.save(model.state_dict(), LAST_CKPT)
        if vl < best_val:
            best_val = vl
            torch.save(model.state_dict(), BEST_CKPT)
            print(f'  → новый лучший val: {best_val:.4f}, сохранено в {BEST_CKPT}')
        with open(HISTORY_JSON, 'w') as f:
            json.dump(history, f, indent=2)
    model.load_state_dict(torch.load(BEST_CKPT, map_location=device, weights_only=True))
    print('Обучение завершено, загружен лучший чекпоинт.')



## Кривые потерь


In [ ]:

with open(HISTORY_JSON) as f:
    history = json.load(f)

epochs = list(range(1, len(history['train_loss']) + 1))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, history['train_loss'], marker='o', label='train')
ax.plot(epochs, history['val_loss'],   marker='s', label='val')
ax.set_xlabel('эпоха')
ax.set_ylabel('cross-entropy loss')
ax.set_title('Кривые потерь (Colab base)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()



## Декодирование

Greedy и beam search — те же, что и локально.


In [ ]:

@torch.no_grad()
def greedy_decode(model, src_text, max_len=80):
    model.eval()
    src_ids = torch.tensor([encode_src(src_text)], device=device)
    src_pm  = (src_ids == PAD_ID)
    memory  = model.encode(src_ids, src_pm)

    ys = torch.tensor([[BOS_ID]], device=device)
    for _ in range(max_len):
        tgt_mask = generate_square_subsequent_mask(ys.size(1), device)
        out = model.decode(ys, memory, tgt_mask,
                           tgt_pad_mask=(ys == PAD_ID),
                           src_pad_mask=src_pm)
        logits = model.proj(out[:, -1])
        nxt = int(logits.argmax(-1).item())
        ys = torch.cat([ys, torch.tensor([[nxt]], device=device)], dim=1)
        if nxt == EOS_ID:
            break

    ids = ys[0].tolist()[1:]
    if ids and ids[-1] == EOS_ID:
        ids = ids[:-1]
    return tok.decode(ids)


In [ ]:

@torch.no_grad()
def beam_search_decode(model, src_text, beam_size=BEAM, max_len=80, length_penalty=0.6):
    model.eval()
    src_ids = torch.tensor([encode_src(src_text)], device=device)
    src_pm  = (src_ids == PAD_ID)
    memory  = model.encode(src_ids, src_pm)

    def norm(tokens, score):
        L = max(len(tokens) - 1, 1)
        return score / (((5 + L) / 6) ** length_penalty)

    beams = [[[BOS_ID], 0.0]]
    finished = []

    for _ in range(max_len):
        if not beams:
            break

        ys = torch.tensor([b[0] for b in beams], device=device)
        B  = ys.size(0)
        mem_b   = memory.expand(B, -1, -1)
        src_pm_b = src_pm.expand(B, -1)
        tgt_mask = generate_square_subsequent_mask(ys.size(1), device)
        out = model.decode(ys, mem_b, tgt_mask,
                           tgt_pad_mask=(ys == PAD_ID),
                           src_pad_mask=src_pm_b)
        log_probs = torch.log_softmax(model.proj(out[:, -1]), dim=-1)
        topk_lp, topk_idx = log_probs.topk(beam_size, dim=-1)

        cands = []
        for i, (tokens, score) in enumerate(beams):
            for k in range(beam_size):
                nxt = int(topk_idx[i, k].item())
                nlp = float(topk_lp[i, k].item())
                cands.append((tokens + [nxt], score + nlp))

        cands.sort(key=lambda c: norm(c[0], c[1]), reverse=True)

        next_beams = []
        for tokens, score in cands:
            if tokens[-1] == EOS_ID:
                finished.append((tokens, score))
            else:
                next_beams.append([tokens, score])
            if len(next_beams) == beam_size:
                break
        beams = next_beams

        if len(finished) >= beam_size:
            break

    pool = finished + [(b[0], b[1]) for b in beams]
    pool.sort(key=lambda c: norm(c[0], c[1]), reverse=True)

    ids = pool[0][0][1:]
    if ids and ids[-1] == EOS_ID:
        ids = ids[:-1]
    return tok.decode(ids)



## Метрика: BLEU

Считаем на 1000 примерах из теста (вдвое больше, чем локально — на L4/A100 это быстро).


In [ ]:

import sacrebleu

def translate_corpus(pairs, decoder_fn, desc):
    hyps, refs = [], []
    for ru, en in tqdm(pairs, desc=desc):
        hyps.append(decoder_fn(ru))
        refs.append(en)
    return hyps, refs

N_BLEU = min(1000, len(test_pairs))
bleu_pairs = test_pairs[:N_BLEU]

g_hyps, refs = translate_corpus(bleu_pairs, lambda t: greedy_decode(model, t), desc='greedy')
bleu_g = sacrebleu.corpus_bleu(g_hyps, [refs])
print(f'GREEDY BLEU @ {N_BLEU}: {bleu_g.score:.2f}')

b_hyps, _ = translate_corpus(bleu_pairs, lambda t: beam_search_decode(model, t, BEAM), desc='beam')
bleu_b = sacrebleu.corpus_bleu(b_hyps, [refs])
print(f'BEAM-{BEAM} BLEU @ {N_BLEU}: {bleu_b.score:.2f}')



## 20 примеров из тестовой выборки

5 коротких + 10 средних + 5 длинных пар, для каждой — `RU / REF / GREEDY / BEAM`.


In [ ]:

def pick_diverse(pairs, n_short=5, n_med=10, n_long=5, seed=SEED):
    short, med, long = [], [], []
    for ru, en in pairs:
        w = len(ru.split())
        if w <= 6:
            short.append((ru, en))
        elif w <= 20:
            med.append((ru, en))
        else:
            long.append((ru, en))
    r = random.Random(seed)
    r.shuffle(short); r.shuffle(med); r.shuffle(long)
    return short[:n_short] + med[:n_med] + long[:n_long]

samples = pick_diverse(test_pairs)
print(f'Подобрано {len(samples)} примеров')
print('=' * 80)
for i, (ru, en) in enumerate(samples, 1):
    g = greedy_decode(model, ru)
    b = beam_search_decode(model, ru, BEAM)
    print(f'[{i:02d}] RU:     {ru}')
    print(f'     REF:    {en}')
    print(f'     GREEDY: {g}')
    print(f'     BEAM:   {b}')
    print('-' * 80)



## Авторские примеры


In [ ]:

authored = [
    'Доброе утро, рад вас видеть.',
    'Слушай, давай уже пойдём, я устал.',
    'Не лезь не в своё дело.',
    'Иван Петрович родился в Москве в 1985 году.',
    'Цена выросла с 1200 до 1500 рублей за килограмм.',
    'Я никогда не говорил, что не люблю этот фильм.',
    'Сколько ещё ждать до отправления поезда?',
    'Хотя погода была плохой, мы всё равно решили пойти в поход и провели прекрасный день у озера.',
    'Этот нейросетевой алгоритм использует механизм внимания для перевода предложений.',
    'Он быстро бросил ключ в замок и открыл дверь.',
]

for i, ru in enumerate(authored, 1):
    b = beam_search_decode(model, ru, BEAM)
    print(f'[{i:02d}] RU: {ru}')
    print(f'     EN: {b}')
    print('-' * 80)



## Интерактивный перевод

Замените `text` и перезапустите ячейку.


In [ ]:

text = 'Сегодня хороший день для машинного перевода.'
print('RU:', text)
print('EN:', beam_search_decode(model, text, BEAM))
